# Generating Fleet Available Capacity Simulator – Extended Project (Solution)

**Energy / Power-Systems application of the Monte-Carlo “sum of independent discrete units” idea**  
(adapted from Al Sweigart’s Million Dice Roll Statistics Simulator, Project #46, and the earlier Economic Loss version)

In power-system reliability analysis we need the probability distribution of the *total available capacity* of a fleet of generating units.  
Each unit can be in one of several discrete capacity states (forced outage, partial derating, or full output).  
The classic industrial method is Monte Carlo simulation of many independent scenarios of the fleet.

This notebook provides:

- Simulation of total available capacity from N independent generating units
- Progress reporting, frequency table and histogram
- Low-tail reliability metrics (capacity VaR-style quantile, Expected Shortfall of capacity, simple LOLP-like probabilities)
- Two alternate implementations (Counter + NumPy vectorized)
- Exact theoretical distribution via dynamic programming
- Parameter sweeps that illustrate the Central Limit Theorem for large fleets
- Configurable “More Practice” and Simulation sections

Use the matching **Practice Skeleton** to implement the pieces yourself first.


In [ ]:
from IPython.display import Image, display
display(Image(filename='generating_fleet_capacity_flowchart.png', width=950))
print("Flowchart: Generating Fleet Available Capacity Simulator (Monte Carlo path + alternates).")


## 0. Imports & Reproducibility


In [ ]:
import random
import time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

print("Libraries loaded.")


## 1. Core Simulation Function

`simulate_fleet_capacity(n_units, states=6, num_simulations=1_000_000, show_progress=True)`

Each of the `n_units` generating units independently realises a discrete capacity level drawn uniformly from 1 … `states` (MW units).  
(The uniform discrete model keeps the parallel with the original dice project transparent while still being a valid multi-state capacity model. Two-state up/down models are explored in the practice section.)

Returns a dictionary: aggregate_capacity → count.


In [ ]:
def simulate_fleet_capacity(n_units, states=6, num_simulations=1_000_000, show_progress=True):
    """Simulate total available capacity of n_units independent multi-state generators.
    Each unit capacity ∈ {1, 2, …, states}. Returns dict: total_capacity → count.
    """
    min_cap = n_units * 1
    max_cap = n_units * states
    results = {c: 0 for c in range(min_cap, max_cap + 1)}

    if show_progress:
        print(f'Simulating {num_simulations:,} scenarios of a fleet of {n_units} units '
              f'(each capacity ∈ {{1..{states}}} MW)...')
    last_print_time = time.time()

    for i in range(num_simulations):
        if show_progress and time.time() > last_print_time + 1:
            pct = round(i / (num_simulations / 100), 1)
            print(f'{pct}% done...')
            last_print_time = time.time()

        total = 0
        for _ in range(n_units):
            total += random.randint(1, states)
        results[total] += 1

    return results


# Demo (small number of sims for responsiveness)
demo = simulate_fleet_capacity(n_units=2, states=6, num_simulations=100_000, show_progress=True)
print("Sample counts (first few):", dict(list(demo.items())[:5]), "...")


## 2. Display Frequency Table + Reliability Metrics

In power-system reliability we care especially about the *lower tail* of available capacity (risk of insufficient generation).

We report:

- Classic TOTAL CAPACITY – COUNT – PERCENTAGE table
- **Capacity VaR_α** – the α-quantile of available capacity (a high-confidence lower bound on fleet output)
- **Expected Shortfall of capacity** – average capacity in the worst (1-α) tail
- Simple **LOLP-style** probability that capacity falls below a chosen threshold


In [ ]:
def display_capacity_and_reliability(results, num_simulations=1_000_000, alpha=0.05, threshold=None):
    """Print frequency table and low-tail reliability metrics.
    alpha is the lower-tail probability (e.g. 0.05 → 5 % worst capacity scenarios).
    """
    print('TOTAL CAPACITY - COUNT - PERCENTAGE')
    for cap in sorted(results.keys()):
        count = results[cap]
        pct = round(count / num_simulations * 100, 1)
        print(f' {cap:3d} - {count:7d} - {pct:5.1f}%')

    # Build ordered list of simulated capacities (memory-friendly from histogram)
    sorted_caps = []
    for cap in sorted(results.keys()):
        sorted_caps.extend([cap] * results[cap])

    # Lower-tail VaR (α-quantile of capacity)
    idx = int(np.floor(alpha * num_simulations))
    capacity_var = sorted_caps[idx]

    # Expected Shortfall of capacity = mean of the worst α-tail
    tail = sorted_caps[:idx+1]
    es_cap = np.mean(tail) if tail else capacity_var

    print(f'\nLow-tail reliability metrics (α = {alpha}):')
    print(f'  Capacity VaR_{alpha:.0%} (lower) = {capacity_var} MW')
    print(f'  ES of capacity (worst {alpha:.0%}) = {es_cap:.2f} MW')

    if threshold is not None:
        lolp = sum(results[c] for c in results if c < threshold) / num_simulations
        print(f'  LOLP-style P(capacity < {threshold}) = {lolp:.4f} ({lolp*100:.2f}%)')

    return capacity_var, es_cap


print("=== Fleet of 2 units, 100 000 scenarios ===")
var_demo, es_demo = display_capacity_and_reliability(demo, 100_000, alpha=0.05, threshold=5)


## 3. Visualization – Available Capacity Distribution

A bar chart of the empirical percentages shows the shape of the fleet capacity distribution.  
For small fleets the distribution is triangular; for large fleets it becomes approximately normal (Central Limit Theorem – the reason many system studies can use normal approximations for large generation portfolios).


In [ ]:
def plot_capacity_distribution(results, n_units, states, num_simulations, title_suffix="", fname='fleet_capacity_distribution.png'):
    caps = sorted(results.keys())
    percentages = [results[c] / num_simulations * 100 for c in caps]

    plt.figure(figsize=(10, 5))
    plt.bar(caps, percentages, color='steelblue', edgecolor='navy', alpha=0.85)
    plt.xlabel('Total Available Capacity (MW units)', fontsize=12)
    plt.ylabel('Percentage (%)', fontsize=12)
    plt.title(f'Empirical fleet available-capacity distribution\n'
              f'{n_units} units × discrete states 1..{states} '
              f'({num_simulations:,} Monte-Carlo scenarios){title_suffix}', fontsize=13)
    plt.xticks(caps[::max(1, len(caps)//15)])
    plt.grid(axis='y', alpha=0.3)

    mode = max(results, key=results.get)
    plt.annotate(f'mode={mode}', xy=(mode, results[mode]/num_simulations*100),
                 xytext=(mode + 0.8, results[mode]/num_simulations*100 + 1.2),
                 arrowprops=dict(arrowstyle='->', color='crimson'), color='crimson')
    plt.tight_layout()
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved as {fname}")


plot_capacity_distribution(demo, 2, 6, 100_000)


## 4. Alternate Implementation #1 – collections.Counter


In [ ]:
def simulate_with_counter(n_units, states=6, num_simulations=1_000_000):
    counter = Counter()
    for _ in range(num_simulations):
        total = sum(random.randint(1, states) for _ in range(n_units))
        counter[total] += 1
    for c in range(n_units, n_units * states + 1):
        counter.setdefault(c, 0)
    return dict(counter)


c_res = simulate_with_counter(2, 6, 50_000)
print("Counter version (50k scenarios) – first few keys:", dict(list(c_res.items())[:4]))
print("Total scenarios accounted for:", sum(c_res.values()))


## 5. Alternate Implementation #2 – NumPy Vectorized (production speed)

Used in real power-system Monte-Carlo reliability engines when the number of scenarios reaches millions.


In [ ]:
def simulate_numpy(n_units, states=6, num_simulations=1_000_000):
    draws = np.random.randint(1, states + 1, size=(num_simulations, n_units))
    totals = draws.sum(axis=1)
    unique, counts = np.unique(totals, return_counts=True)
    results = {int(u): int(c) for u, c in zip(unique, counts)}
    for c in range(n_units, n_units * states + 1):
        results.setdefault(c, 0)
    return results


t0 = time.time()
_ = simulate_fleet_capacity(5, 6, 200_000, show_progress=False)
t_loop = time.time() - t0
t0 = time.time()
_ = simulate_numpy(5, 6, 200_000)
t_np = time.time() - t0
print(f"Pure-Python loop 200k scenarios of 5 units: {t_loop:.3f}s")
print(f"NumPy vectorized                         : {t_np:.3f}s")
print(f"Speed-up ≈ {t_loop / max(t_np, 1e-9):.1f}×")


## 6. Exact Theoretical Distribution (Dynamic Programming)

When each unit is discrete uniform on {1…S} the exact PMF of total fleet capacity is obtained by the classic recurrence used for dice sums.  
This provides a gold-standard benchmark for the Monte-Carlo approximation on small fleets.


In [ ]:
def theoretical_probs(n_units, states=6):
    max_sum = n_units * states
    dp = [0] * (max_sum + 1)
    dp[0] = 1
    for k in range(1, n_units + 1):
        new_dp = [0] * (max_sum + 1)
        for prev, ways in enumerate(dp):
            if ways == 0:
                continue
            for face in range(1, states + 1):
                new_dp[prev + face] += ways
        dp = new_dp
    total_ways = states ** n_units
    return {s: dp[s] / total_ways for s in range(n_units, max_sum + 1) if dp[s] > 0}


theo = theoretical_probs(2, 6)
print("Exact probabilities for a 2-unit fleet (states 1..6):")
for cap, p in sorted(theo.items()):
    print(f"  capacity={cap}: {p*100:.1f}%  ({int(round(p*36))}/36)")


## 7. Empirical vs Theoretical Comparison


In [ ]:
def compare_emp_theo(n_units=2, states=6, num_simulations=500_000, alpha=0.05):
    emp = simulate_numpy(n_units, states, num_simulations)
    theo = theoretical_probs(n_units, states)

    caps = sorted(emp.keys())
    emp_pct = [emp[c] / num_simulations * 100 for c in caps]
    theo_pct = [theo.get(c, 0) * 100 for c in caps]

    plt.figure(figsize=(10, 5))
    width = 0.4
    x = np.array(caps)
    plt.bar(x - width/2, emp_pct, width, label='Empirical (Monte-Carlo)', color='steelblue', alpha=0.8)
    plt.bar(x + width/2, theo_pct, width, label='Theoretical (exact)', color='darkorange', alpha=0.8)
    plt.xlabel('Total Available Capacity')
    plt.ylabel('Percentage (%)')
    plt.title(f'{n_units}-unit fleet: Empirical ({num_simulations:,}) vs Exact')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('fleet_capacity_emp_vs_theo.png', dpi=120, bbox_inches='tight')
    plt.show()

    max_err = max(abs(e - t) for e, t in zip(emp_pct, theo_pct))
    print(f"Max absolute percentage error: {max_err:.3f} pp")

    # Empirical lower-tail capacity VaR from the same run
    sorted_caps = []
    for c in caps:
        sorted_caps.extend([c] * emp[c])
    idx = int(np.floor(alpha * num_simulations))
    print(f"Empirical Capacity VaR_{alpha:.0%} (lower) from this run: {sorted_caps[idx]}")


compare_emp_theo(2, 6, 500_000)


## 8. More Practice Exercises

1. Simulate 200 000 scenarios of a **5-unit fleet** and report mode, Capacity VaR_5 % and a LOLP-style probability below a chosen threshold.  
2. Model a **two-state** unit (available at full capacity or forced outage = 0). You can approximate this by restricting the support or by a simple Bernoulli draw.  
3. A fleet of **10 identical binary units** (on/off). Examine the distribution of the number of units that are available.  
4. Use the theoretical function to confirm the classic distribution of two units with support size 2.


In [ ]:
# Practice 1 – 5-unit fleet
print("=== Practice 1: 5-unit fleet (200 k scenarios) ===")
res5 = simulate_numpy(5, 6, 200_000)
display_capacity_and_reliability(res5, 200_000, alpha=0.05, threshold=15)
print(f"Mode ≈ {max(res5, key=res5.get)}")

# Practice 3 – 10 binary units
print("\n=== Practice 3: 10 binary (on/off) units ===")
res_bin = simulate_numpy(10, 2, 100_000)
display_capacity_and_reliability(res_bin, 100_000, alpha=0.05, threshold=4)

# Practice 4 – theoretical two binary units
print("\n=== Practice 4: exact distribution of two binary units ===")
theo_bin = theoretical_probs(2, 2)
print({k: f'{v*100:.1f}%' for k, v in theo_bin.items()})


## 9. Simulation Section – Parameter Sweeps & CLT for Large Fleets

System planners care how the shape of the available-capacity distribution changes as the number of independent units grows.  
By the Central Limit Theorem the standardised sum converges to a normal distribution – this underpins many analytic approximations used for large generation portfolios.

Edit the configuration list and re-run to explore different fleet sizes.


In [ ]:
def run_fleet_sweep(configs):
    """configs = list of (n_units, states, n_sims, label)"""
    fig, axes = plt.subplots(1, len(configs), figsize=(5*len(configs), 4), sharey=True)
    if len(configs) == 1:
        axes = [axes]

    for ax, (n, s, sims, label) in zip(axes, configs):
        res = simulate_numpy(n, s, sims)
        caps = sorted(res.keys())
        pct = [res[c]/sims*100 for c in caps]
        ax.bar(caps, pct, color='teal', alpha=0.8, edgecolor='darkgreen')
        mean = sum(c * res[c] for c in caps) / sims
        ax.axvline(mean, color='red', linestyle='--', label=f'mean≈{mean:.1f}')
        ax.set_title(label)
        ax.set_xlabel('Available Capacity')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    axes[0].set_ylabel('Percentage (%)')
    plt.suptitle('Fleet available-capacity distributions – Central Limit Theorem in power systems', fontsize=13)
    plt.tight_layout()
    plt.savefig('fleet_capacity_simulation_sweep.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Sweep chart saved as fleet_capacity_simulation_sweep.png")


experiments = [
    (1, 6, 50_000, '1 unit (flat)'),
    (2, 6, 100_000, '2 units (triangle)'),
    (5, 6, 200_000, '5 units (≈normal)'),
    (15, 6, 200_000, '15 units (very normal)'),
]
run_fleet_sweep(experiments)


## 10. Full-Scale Classic-Style Run (optional)

Uncomment the cell below to run a full 1 000 000-scenario simulation of a small fleet.  
Progress messages appear once per second, exactly as in the original book program.


In [ ]:
# Uncomment for the authentic million-scenario run:
# full = simulate_fleet_capacity(2, states=6, num_simulations=1_000_000, show_progress=True)
# display_capacity_and_reliability(full, 1_000_000, alpha=0.05, threshold=5)
# plot_capacity_distribution(full, 2, 6, 1_000_000, title_suffix=" – full 1 M scenarios")

print("Full-million cell is commented out for notebook responsiveness.")
print("Uncomment when you want the classic slow progress output.")


## Key Takeaways (Energy / Power Systems)

- Monte Carlo is the industrial standard for obtaining the distribution of available capacity of a generating fleet when units have discrete (or discretised) states.
- Low-tail metrics (Capacity VaR, Expected Shortfall of capacity, LOLP-style probabilities) are obtained directly from the ordered simulated scenarios.
- The Central Limit Theorem explains why large fleets of independent units produce approximately normal available-capacity distributions – the foundation of many analytic reliability approximations.
- Vectorized (NumPy) implementations turn a pedagogical loop into production-speed code used in real reliability engines.
- Always benchmark Monte-Carlo results against an exact DP solution for small fleets; the discrepancy quantifies sampling error.
- The same procedure extends immediately to two-state (up/down) units, multi-state deratings, or correlated weather-driven renewable fleets.
